The primary objective of this notebook is to execute Multi-Cohort Feature Alignment and Matrix Integration across independent small RNA-Seq datasets. Because the raw data spans three distinct collection years (2020, 2022, and 2023), individual spreadsheets exhibit variations in sample counts, sequencing depth, and detected microRNA (miRNA) features.

In [16]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/PD ISEF project'
RAW_DIR = os.path.join(PROJECT_PATH, 'data/raw/')
PROCESSED_DIR = os.path.join(PROJECT_PATH, 'data/processed/')

print("[+] Cell 1 Complete: Drive connected and paths locked in!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[+] Cell 1 Complete: Drive connected and paths locked in!


In [17]:
file_path = os.path.join(RAW_DIR, "GSE269775_raw_counts.xlsx")
df = pd.read_excel(file_path, header=1)
print(df.columns[:5])

Index(['Geneid', 'Chr', 'Start', 'End', 'Strand'], dtype='object')


In [18]:
# 1. Point to file 776
path_776 = os.path.join(RAW_DIR, "GSE269776_raw_counts.xlsx")

# 2. Try to read the hidden 'matome' tab directly
try:
    df_776_test = pd.read_excel(path_776, sheet_name="matome")
    print("[+] Found 'matome' tab! Here is the top:")
    print(df_776_test.iloc[:5, :5])
except Exception as e:
    print(f"[X] Could not read 'matome' sheet: {e}")

[X] Could not read 'matome' sheet: Worksheet named 'matome' not found


In [19]:
master_gzip_path = os.path.join(RAW_DIR, 'GSE269781_family.soft.gz')
import gzip

# Open the file in text-reading mode ('rt')
with gzip.open(master_gzip_path, 'rt', encoding='utf-8') as f:
    # Read just the first 50 lines to see how the data structure starts
    for i in range(50):
        print(f.readline().strip())

^DATABASE = GeoMiame
!Database_name = Gene Expression Omnibus (GEO)
!Database_institute = NCBI NLM NIH
!Database_web_link = http://www.ncbi.nlm.nih.gov/geo
!Database_email = geo@ncbi.nlm.nih.gov
^SERIES = GSE269781
!Series_title = Comprehensive data for studying serum exosome microRNA transcriptome in Parkinson's disease patients
!Series_geo_accession = GSE269781
!Series_status = Public on Jul 26 2024
!Series_submission_date = Jun 13 2024
!Series_last_update_date = Oct 16 2024
!Series_pubmed_id = 39406833
!Series_web_link = 10.1038/s41597-024-03909-6
!Series_summary = This SuperSeries is composed of the SubSeries listed below.
!Series_overall_design = Refer to individual Series
!Series_type = Non-coding RNA profiling by high throughput sequencing
!Series_sample_id = GSM8326771
!Series_sample_id = GSM8326772
!Series_sample_id = GSM8326773
!Series_sample_id = GSM8326774
!Series_sample_id = GSM8326775
!Series_sample_id = GSM8326776
!Series_sample_id = GSM8326777
!Series_sample_id = GSM832

In [20]:
f = gzip.open(master_gzip_path, 'rt', encoding='utf-8')
goodones = []
for line_num, line in enumerate(f):
  if line.startswith('^SAMPLE'):
    goodones.append(line_num)

print(goodones[:15])

[444, 486, 528, 570, 612, 654, 696, 738, 780, 822, 864, 906, 948, 990, 1032]


AHA look the difference is always 42 meaning each sample has 42 lines of metadata behind it meaning we can bypass the corrupted code and extract the data from the backend

In [21]:
import gzip

# Open the compressed master text file
with gzip.open(master_gzip_path, 'rt', encoding='utf-8') as f:
    for line_num, line in enumerate(f):
        # Print only the lines within our first patient block window
        if 444 <= line_num <= 485:
            print(f"{line_num}: {line.strip()}")
        # Stop reading after the window to save time and memory
        elif line_num > 485:
            break

444: ^SAMPLE = GSM8326771
445: !Sample_title = Serum exosome miRNA, PD, 2020, 01
446: !Sample_geo_accession = GSM8326771
447: !Sample_status = Public on Jul 26 2024
448: !Sample_submission_date = Jun 13 2024
449: !Sample_last_update_date = Jul 26 2024
450: !Sample_type = SRA
451: !Sample_channel_count = 1
452: !Sample_source_name_ch1 = Serum exosome
453: !Sample_organism_ch1 = Homo sapiens
454: !Sample_taxid_ch1 = 9606
455: !Sample_characteristics_ch1 = tissue: Serum exosome
456: !Sample_characteristics_ch1 = disease state: PD
457: !Sample_characteristics_ch1 = cell type: Extracellular vesicles
458: !Sample_treatment_protocol_ch1 = Serum samples were collected from participants at the outpatient department of Juntendo University Hospital. Upon collection, the serum samples were immediately stored at −80°C until use.
459: !Sample_molecule_ch1 = total RNA
460: !Sample_extract_protocol_ch1 = Total Exosome RNA and Protein Isolation Kit (Thermo Fisher Scientific, Waltham, MA) was utilised t

In [22]:
import os
import pandas as pd

path_776 = os.path.join(RAW_DIR, "GSE269776_raw_counts.xlsx")

# Read the sheet directly
df_776_test = pd.read_excel(path_776, sheet_name="CountMatrix")

print("Columns found in CountMatrix:")
print(list(df_776_test.columns[:5]))

print("\nRow 5 to 10 of the first sample column:")
print(df_776_test.iloc[5:11, 1])

print("\nData type of these cells:")
print(type(df_776_test.iloc[5, 1]))

Columns found in CountMatrix:
['Unnamed: 0', 'J1', 'J2', 'J3', 'J4']

Row 5 to 10 of the first sample column:
5                              NaN
6     PD-01_IonXpressRNA_012.fastq
7                          #ERROR!
8                          #ERROR!
9                          #ERROR!
10                         #ERROR!
Name: J1, dtype: object

Data type of these cells:
<class 'float'>


I discovered that the 2021 cohort was completely corrupted so that complicates the whole process because now I have to build around that.

In [23]:
# Print the first 30 rows and all columns to see the layout map
print("--- Layout Scan (Rows 0 to 30) ---")
print(df_776_test.iloc[:30, :])

--- Layout Scan (Rows 0 to 30) ---
                        Unnamed: 0                            J1  \
0                          RPM>0.1                             0   
1                            RPM>1                             0   
2                           RPM>10                             0   
3                          RPM>100                             0   
4                         RPM>1000                             0   
5                              NaN                           NaN   
6                           Geneid  PD-01_IonXpressRNA_012.fastq   
7   hsa-mir-6859-1_hsa-miR-6859-5p                       #ERROR!   
8   hsa-mir-6859-1_hsa-miR-6859-3p                       #ERROR!   
9      hsa-mir-1302-2_hsa-miR-1302                       #ERROR!   
10    hsa-mir-6723_hsa-miR-6723-5p                       #ERROR!   
11    hsa-mir-200b_hsa-miR-200b-5p                       #ERROR!   
12    hsa-mir-200b_hsa-miR-200b-3p                       #ERROR!   
13    hsa-mir

In [24]:
import os

# Find all series matrix files in your raw folder
matrix_files = [f for f in os.listdir(RAW_DIR) if "series_matrix" in f]
print("Available Series Matrix files:")
for f in matrix_files:
    print(f"- {f}")

Available Series Matrix files:
- GSE269776_series_matrix.txt
- GSE269781_series_matrix.txt


In [ ]:
import os
import numpy as np
import pandas as pd

file_2020 = os.path.join(RAW_DIR, 'GSE269775_raw_counts.xlsx')
file_2022 = os.path.join(RAW_DIR, 'GSE269777_raw_counts.xlsx')
file_2023 = os.path.join(RAW_DIR, 'GSE269779_raw_counts.xlsx')

df_2020_raw = pd.read_excel(file_2020, header=1)
unwanted = ['Chr', 'Start', 'End', 'Strand', 'Length']#unwanted metadata
df_2020_clean = df_2020_raw.drop(columns=unwanted).set_index('Geneid')

df_2022_clean = pd.read_excel(file_2022).set_index('Gene')
df_2023_clean = pd.read_excel(file_2023).set_index('Gene')

In [ ]:
df_merged = pd.concat([df_2020_clean, df_2022_clean, df_2023_clean], join='inner', axis=1)

column_totals = df_merged.sum()
df_cpm = (df_merged / column_totals) * 1e6

df_filtered = df_cpm[(df_cpm >= 1).sum(axis=1) >= 32]
df_log2 = np.log2(df_filtered + 1)
X_raw_log = df_log2.T.iloc[:319]

print("[-] Verifying aligned matrix configurations...")
print(f"-> Total Sample Instances: {X_raw_log.shape[0]}")
print(f"-> Total Retained Genetic Features: {X_raw_log.shape[1]}")

In [ ]:
output_path = os.path.join(PROCESSED_DIR, "clean_normalized_counts.csv")
X_raw_log.to_csv(output_path)
print(output_path)